In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


# CpG content and methylation of CENP-A repeat arrays, by TRF period bin

For each of the **9 TRF period bins** used in the CENP-A period-enrichment analysis, this
notebook computes three per-array metrics over the **merged repeat arrays** (`bedtools
merge -d 0` of TRF intervals within a bin; never across bins — see `scripts/1b_merge_arrays.sh`):

| # | Metric | Definition |
|---|--------|-----------|
| 1 | **Average array length (bp)** | mean over arrays of `end − start` |
| 2 | **Average CpG sites / 1000 bp** | mean over arrays of reference CG-dinucleotide density: `1000 × CG / length` (case-insensitive count on the assembly sequence) |
| 3 | **CpG methylation fraction** | mean over arrays (with ≥ 1 callable CpG) of the per-array mean WGBS fraction-methylated at CpGs with read coverage ≥ 5 |

**Bin schema** (period = TRF consensus repeat length): `1–10 bp`, `11–50 bp`, `51–192 bp`,
`193–195 bp`, `196–347 bp`, **`348–349 bp` (the degu centromeric satellite — bin 6)**,
`350–385 bp`, `386–390 bp`, `391+ bp`.

**Data sources**

- Merged arrays: `period-enrichment/data/merged/arrays/bin<N>_arrays.bed`
- Assembly (CpG counting): `data/denovo_OctDegus_genome/041425-assembly/…/assembly_final.sorted.headerRenamed.chrAssigned.contamFiltered.fasta` (soft-masked — counts are case-insensitive)
- WGBS (PFC, CGN context, both strands):
  `circos-plot/feature-overview/degu_6834_PFC_1.CGN-both.frac.bw` (fraction methylated per CpG) and
  `…CGN-both.cov.bw` (read coverage per CpG)
- Methylation filtering: `MIN_COV = 5` (same threshold as the CENP-A-core methylation metrics in `cenpa-repeat-chromosome/scripts/7_methylation_cpg_metrics.py`)


In [ ]:
# ============================================================================
# Setup: libraries, paths, constants
# ============================================================================
%matplotlib inline

import os
import numpy as np
import pandas as pd
import pyBigWig
import pyfaidx

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- project paths ----------------------------------------------------------
FIGURE_DIR = f"{PROJ_ROOT}/figure/cenpa-cuttag-enrichment"
PERIOD_DIR = os.path.join(FIGURE_DIR, "period-enrichment")
ARRAY_DIR  = os.path.join(PERIOD_DIR, "data", "merged", "arrays")
RESULT_DIR = os.path.join(PERIOD_DIR, "results")
PLOT_DIR   = os.path.join(PERIOD_DIR, "plots")
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

ASSEMBLY_FASTA = (f"{PROJ_ROOT}/data/denovo_OctDegus_genome/"
                  "041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-"
                  "chrNameAssigned-contamFiltered/assembly_final.sorted.headerRenamed.chrAssigned.contamFiltered.fasta")
BW_DIR  = f"{PROJ_ROOT}/figure/circos-plot/feature-overview"
BW_FRAC = os.path.join(BW_DIR, "degu_6834_PFC_1.CGN-both.frac.bw")
BW_COV  = os.path.join(BW_DIR, "degu_6834_PFC_1.CGN-both.cov.bw")

MIN_COV  = 5          # callable-CpG coverage threshold (same as 7_methylation_cpg_metrics.py)
N_BINS   = 9

BIN_LABEL = {1: "1-10 bp",  2: "11-50 bp",  3: "51-192 bp",
             4: "193-195 bp", 5: "196-347 bp", 6: "348-349 bp",
             7: "350-385 bp", 8: "386-390 bp", 9: "391+ bp"}

for p, label in [(ARRAY_DIR, "array dir"), (ASSEMBLY_FASTA, "assembly fasta"),
                 (BW_FRAC, "WGBS frac bigwig"), (BW_COV, "WGBS cov bigwig")]:
    assert os.path.exists(p), f"missing {label}: {p}"
print("setup OK")
print("figure dir:", FIGURE_DIR)


## 1. Load merged arrays and compute array length

The arrays are already merged (`-d 0`) per bin and non-overlapping within a bin, so
`length = end − start` is well defined. Bin 6 (the 348–349 bp satellite) is dominated by
a few very long arrays — the *mean* array length is therefore much larger there than in
any other bin.

In [ ]:
# ============================================================================
# 1. Load merged arrays per bin, compute per-array length
# ============================================================================
frames = []
for b in range(1, N_BINS + 1):
    df = pd.read_csv(os.path.join(ARRAY_DIR, f"bin{b}_arrays.bed"), sep="\t", header=None,
                     names=["chrom", "start", "end", "array_id", "n_intervals"])
    df["bin_id"] = b
    df["length"] = df["end"] - df["start"]
    frames.append(df)
arr = pd.concat(frames, ignore_index=True)

print(f"{len(arr):,} merged arrays total")
arr.groupby("bin_id")["length"].describe()[["count", "mean", "50%", "max"]]

## 2. Reference CpG sites per 1000 bp

CpG density is computed from the **assembly sequence** (case-insensitive `CG`
dinucleotide count, so soft-masked repeats still count). This is the *potential* CpG
content of each array, independent of WGBS coverage — the same quantity the
CENP-A-core metrics call `ref_CG` / `cg_density_per_kb`.

> Performance: one full-chromosome fetch per chromosome, then `str.count` per array
> (~200 Mb of array sequence total) — a few seconds.

In [ ]:
# ============================================================================
# 2. Reference CG-dinucleotide density per array (case-insensitive)
# ============================================================================
fa = pyfaidx.Fasta(ASSEMBLY_FASTA, as_raw=True)

ref_cg = np.empty(len(arr), dtype=np.int64)
for chrom, grp in arr.groupby("chrom", sort=False):
    seq = fa[chrom][:].upper()                      # full chromosome, upper-cased
    counts = np.array([seq[s:e].count("CG")
                       for s, e in zip(grp["start"].values, grp["end"].values)])
    ref_cg[grp.index] = counts
arr["ref_CG"] = ref_cg
arr["cg_density_per_kb"] = 1000 * arr["ref_CG"] / arr["length"]
fa.close()

mean_a  = arr.groupby("bin_id")["cg_density_per_kb"].mean().rename("mean_per_array_CpG_per_kb")
pooled  = arr.groupby("bin_id").apply(
    lambda g: 1000 * g["ref_CG"].sum() / g["length"].sum()).rename("pooled_CpG_per_kb")
pd.concat([mean_a, pooled], axis=1).round(2)

## 3. CpG methylation fraction (WGBS)

For each array, the per-CpG fraction-methylated values (`…frac.bw`) are averaged over the
array's **callable** CpGs (`cov ≥ 5` in `…cov.bw`). Arrays with no callable CpG get `NaN`.

Implementation is vectorized per (bin, chromosome): the frac/cov bigwigs are read once per
chromosome (they share aligned CpG positions), each CpG is assigned to its containing array
via `searchsorted` (arrays within a bin are non-overlapping, so assignment is unique), and
per-array `n_cov0`, `n_cov5`, `mean_frac` are reduced with `np.bincount`.

> Note: only a small fraction of reference CpGs in satellite arrays are covered by the WGBS
> reads (bin 6 callable fraction ≈ a few %), so `mean_frac` for bin 6 reflects the
> covered subset — but that subset is large in absolute terms (tens of thousands of CpGs).

In [ ]:
# ============================================================================
# 3. Per-array WGBS methylation fraction (vectorized per bin x chromosome)
# ============================================================================
fr = pyBigWig.open(BW_FRAC)
cv = pyBigWig.open(BW_COV)

# --- read per-chromosome CpG positions once --------------------------------
cpg = {}   # chrom -> (pos, frac, cov), aligned & sorted
for chrom in arr["chrom"].unique():
    fi = fr.intervals(chrom) or []
    ci = cv.intervals(chrom) or []
    pos  = np.fromiter((x[0] for x in fi), dtype=np.int64, count=len(fi))
    frac = np.fromiter((x[2] for x in fi), dtype=np.float64, count=len(fi))
    covd = {int(x[0]): x[2] for x in ci}
    cov  = np.fromiter((covd.get(int(p), np.nan) for p in pos), dtype=np.float64, count=len(pos))
    cpg[chrom] = (pos, frac, cov)
fr.close(); cv.close()

n_cov0    = np.zeros(len(arr), dtype=np.int64)
n_cov5    = np.zeros(len(arr), dtype=np.int64)
mean_frac = np.full(len(arr), np.nan)

for b in range(1, N_BINS + 1):
    sub = arr[arr["bin_id"] == b]
    for chrom, grp in sub.groupby("chrom", sort=False):
        grp = grp.sort_values("start")                      # defensive: ensure sorted
        S, E = grp["start"].values.astype(np.int64), grp["end"].values.astype(np.int64)
        pos, frac, cov = cpg[chrom]
        idx = np.searchsorted(S, pos, side="right") - 1     # containing-array index per CpG
        valid = (idx >= 0) & (pos < E[idx])
        i_ok = idx[valid]
        if len(i_ok) == 0:
            continue
        n0 = np.bincount(i_ok, minlength=len(grp))
        ok5 = valid & (cov >= MIN_COV)
        i5 = idx[ok5]
        n5 = np.bincount(i5, minlength=len(grp))
        sf = np.bincount(i5, weights=frac[ok5], minlength=len(grp))
        with np.errstate(invalid="ignore", divide="ignore"):
            mf = np.where(n5 > 0, sf / n5, np.nan)
        arr.loc[grp.index, "n_cov0"]    = n0
        arr.loc[grp.index, "n_cov5"]    = n5
        arr.loc[grp.index, "mean_frac"] = mf

print("callable CpGs per bin (sum n_cov5):")
print(arr.groupby("bin_id")["n_cov5"].sum().astype(int).to_string())
print("\narrays with >=1 callable CpG:")
print(arr.groupby("bin_id")["n_cov5"].apply(lambda x: int((x > 0).sum())).to_string())

## 4. Per-bin summary table

Averaging the per-array values **within each bin** (the merged array is the unit of
observation, as in the CENP-A enrichment analysis) gives one row per bin:

- `avg_array_length_bp` — mean array length
- `avg_CpG_per_kb` — mean over arrays of reference CpG density
- `avg_methylation_frac` — mean over arrays with ≥ 1 callable CpG of per-array mean fraction methylated
- supporting columns: pooled CpG density, pooled methylation fraction, callable-CpG
  density, number of arrays, and the fraction of arrays contributing a methylation value.

In [ ]:
# ============================================================================
# 4. Per-bin summary: avg array length, avg CpG/kb, avg methylation frac
# ============================================================================
g = arr.groupby("bin_id")
summary = pd.DataFrame({
    "bin":                   list(BIN_LABEL.keys()),
    "bin_label":             [BIN_LABEL[b] for b in BIN_LABEL],
    "n_arrays":              g["length"].count().values,
    "array_bp":              g["length"].sum().values,
    "avg_array_length_bp":   g["length"].mean().round(1).values,
    "median_array_length_bp": g["length"].median().astype(int).values,
    "avg_CpG_per_kb":        g["cg_density_per_kb"].mean().round(2).values,
    "pooled_CpG_per_kb":     g.apply(lambda x: 1000 * x["ref_CG"].sum() / x["length"].sum()).round(2).values,
    "n_CpG_callable":        g["n_cov5"].sum().astype(int).values,
    "callable_CpG_per_kb":   g.apply(lambda x: 1000 * x["n_cov5"].sum() / x["length"].sum()).round(2).values,
    "n_arrays_w_cpg":        g["n_cov5"].apply(lambda x: int((x > 0).sum())).values,
    "frac_arrays_w_cpg":     g["n_cov5"].apply(lambda x: round((x > 0).mean(), 3)).values,
    "avg_methylation_frac":  g.apply(lambda x: x.loc[x["n_cov5"] > 0, "mean_frac"].mean()).round(3).values,
    "pooled_methylation_frac": g.apply(lambda x: (x["mean_frac"] * x["n_cov5"]).sum()
                                       / x["n_cov5"].sum()).round(3).values,
})
display(summary)

summary.to_csv(os.path.join(RESULT_DIR, "period_array_cpg_bin_summary.csv"), index=False)
arr.to_csv(os.path.join(RESULT_DIR, "period_array_cpg_per_array.tsv.gz"),
           sep="\t", index=False, compression="gzip")
print("saved:", os.path.join(RESULT_DIR, "period_array_cpg_bin_summary.csv"))
print("saved:", os.path.join(RESULT_DIR, "period_array_cpg_per_array.tsv.gz"))

## 5. Figure: the three headline metrics across the 9 period bins

In [ ]:
# ============================================================================
# 5. Small-multiples bar figure (single hue; log axis for array length)
# ============================================================================
INK   = "#0b0b0b"
GRID  = "#e1e0d9"
BAR   = "#2a78d6"        # blue (categorical slot 1)
AXIS  = "#c3c2b7"

labels = summary["bin_label"].tolist()

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.3))
fig.suptitle("CENP-A repeat arrays by TRF period bin", fontsize=12, color=INK)

panels = [
    ("Average array length (bp)",          "avg_array_length_bp", "log"),
    ("Average CpG sites / 1000 bp",        "avg_CpG_per_kb",      "linear"),
    ("CpG methylation fraction",           "avg_methylation_frac","linear"),
]

for ax, (title, col, scale) in zip(axes, panels):
    vals = summary[col].to_numpy()
    x = np.arange(len(vals))
    ax.bar(x, vals, width=0.72, color=BAR)
    ax.set_yscale(scale)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_title(title, fontsize=10, color=INK)
    ax.yaxis.grid(True, color=GRID, linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(AXIS)
    ax.tick_params(colors=INK, labelsize=8)
    if scale == "log":
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    # annotate bin 6 (centromeric satellite) on the methylation panel
    if col == "avg_methylation_frac":
        ax.annotate("348-349 bp\nsatellite", xy=(5, vals[5]), xytext=(5, vals[5] + 0.15),
                    ha="center", fontsize=8, color="#e34948",
                    arrowprops=dict(arrowstyle="->", color="#e34948", lw=1.0))

fig.tight_layout(rect=(0, 0, 1, 0.93))
for ext in ("png", "pdf", "svg"):
    fig.savefig(os.path.join(PLOT_DIR, f"period_array_cpg_summary.{ext}"),
                dpi=200, bbox_inches="tight")
plt.show()
print("saved:", os.path.join(PLOT_DIR, "period_array_cpg_summary.{png,pdf,svg}"))

## Reading the results

1. **Array length** — increases with period up to **bin 6 (348–349 bp satellite, mean
   ~142 kb)**, then drops for bins 7–9. The centromeric satellite forms the largest
   continuous repeat arrays in the genome.
2. **CpG density** — highest in the **193–195 bp** and **386–390 bp** bins (~50–56
   CpG/kb), lowest in the 1–10 bp microsatellite bin (~3 CpG/kb).
3. **CpG methylation** — lowest in the **348–349 bp centromeric satellite (~0.48)**,
   consistent with the hypomethylated state of constitutive centromeric heterochromatin;
   highest in the 386–390 bp bin (~0.81–0.83).

Caveats: `avg_methylation_frac` weights each array equally, `pooled_methylation_frac`
weights by callable-CpG count; they agree in direction. The WGBS covered only a few % of
the satellite's reference CpGs (low mappability / coverage in long arrays), so bin 6's
methylation estimate rests on the covered subset (still tens of thousands of CpGs).

Outputs:

- Per-bin summary: `period-enrichment/results/period_array_cpg_bin_summary.csv`
- Per-array cache: `period-enrichment/results/period_array_cpg_per_array.tsv.gz`
- Figure: `period-enrichment/plots/period_array_cpg_summary.{png,pdf,svg}`